In [1]:
import shutil
import os

project_folder = "/kaggle/working/Violence-detection"  # or your project folder path

# Check if it exists first
if os.path.exists(project_folder):
    shutil.rmtree(project_folder)
    print(f"Deleted project folder: {project_folder}")
else:
    print("Project folder does not exist.")

Project folder does not exist.


# 01_data_preparation_and_dataset_pipeline

In [2]:
import os
import shutil

# -----------------------------
# Base project path
# -----------------------------
BASE_DIR = "/kaggle/working/Violence-detection"
os.makedirs(BASE_DIR, exist_ok=True)

# -----------------------------
# Folder structure
# -----------------------------
folders = [
    "data/videos/train/Violence",
    "data/videos/train/NonViolence",
    "data/videos/val/Violence",
    "data/videos/val/NonViolence",
    "data/videos/test/Violence",
    "data/videos/test/NonViolence",
    "notebooks",
    "src/utils",
    "src/dataset",
    "outputs",
    "checkpoints",
    "figures",
    "logs",
    "app",
    "configs"
]

for f in folders:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

print("Folders, scripts!")

Folders, scripts!


## configs/config.py

In [3]:
import os
BASE_DIR = "/kaggle/working/Violence-detection"
os.makedirs(BASE_DIR, exist_ok=True)
# -----------------------------
# Create config.py
# -----------------------------
config_content = '''# Configuration for Violence Detection Project

import os

# =====================
# Paths
# =====================

BASE_DIR = "/kaggle/working/Violence-detection"
DATA_DIR = os.path.join(BASE_DIR, "data")
VIDEOS_DIR = os.path.join(DATA_DIR, "videos")

# =====================
# Video preprocessing
# =====================

NUM_FRAMES = 16
FRAME_SIZE = (224, 224)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# =====================
# Dataset split
# =====================

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

# =====================
# DataLoader
# =====================

BATCH_SIZE = 4
NUM_WORKERS = 0
'''

with open(os.path.join(BASE_DIR, "configs/config.py"), "w") as f:
    f.write(config_content)
    
print("config file is ready!")


config file is ready!


In [4]:
import sys, os

BASE_DIR = "/kaggle/working/Violence-detection"

sys.path.append(BASE_DIR)  #  this is key
sys.path.append(os.path.join(BASE_DIR, "src"))

from configs.config import VIDEOS_DIR, NUM_FRAMES

## Phase 1 Notebook

In [5]:
import os
import shutil
import glob
from configs.config import BASE_DIR ,VIDEOS_DIR,  TRAIN_RATIO, VAL_RATIO, TEST_RATIO, RANDOM_SEED
import random
# 01_data_preparation_and_dataset_pipeline.ipynb
# Phase 1: Dataset Splitting and Preparation

# ----------------------------
# 0️ Setup
# ---------------------------- for reproducibility
random.seed(RANDOM_SEED)

# ----------------------------
# 1️ Local merged dataset folder
# ------------------- ---------
# Suppose you merged all videos in:
LOCAL_MERGED_DIR = "/kaggle/input/datasets/messaoudiriham/merged-violence-dataset/Violence Detection Dataset"

# The folder structure is assumed like this:
# merged_dataset/Violence/*.mp4 or *.avi
# merged_dataset/NonViolence/*.mp4 or *.avi

violence_videos = [os.path.join(LOCAL_MERGED_DIR, "Violence", f)
                    for f in os.listdir(os.path.join(LOCAL_MERGED_DIR, "Violence"))
                    if f.lower().endswith((".mp4", ".avi"))]

nonviolence_videos = [os.path.join(LOCAL_MERGED_DIR, "NonViolence", f)
                      for f in os.listdir(os.path.join(LOCAL_MERGED_DIR, "NonViolence"))
                      if f.lower().endswith((".mp4", ".avi"))]

print(f"Violence videos: {len(violence_videos)}")
print(f"NonViolence videos: {len(nonviolence_videos)}")

# ----------------------------
# 2️ Shuffle and split
# ----------------------------
def split_list(videos, train_ratio, val_ratio, test_ratio):
    random.shuffle(videos)
    n_total = len(videos)
    n_train = int(n_total * train_ratio)
    n_val   = int(n_total * val_ratio)
    n_test  = n_total - n_train - n_val
    train = videos[:n_train]
    val   = videos[n_train:n_train+n_val]
    test  = videos[n_train+n_val:]
    return train, val, test

violence_train, violence_val, violence_test = split_list(violence_videos, TRAIN_RATIO, VAL_RATIO, TEST_RATIO)
nonviolence_train, nonviolence_val, nonviolence_test = split_list(nonviolence_videos, TRAIN_RATIO, VAL_RATIO, TEST_RATIO)

# ----------------------------
# 3️ Copy to structured folders
# ----------------------------
def copy_videos(video_list, split_name, label_name):
    dest_dir = os.path.join(VIDEOS_DIR, split_name, label_name)
    os.makedirs(dest_dir, exist_ok=True)
    for src in video_list:
        shutil.copy(src, dest_dir)

# Violence
copy_videos(violence_train, "train", "Violence")
copy_videos(violence_val,   "val",   "Violence")
copy_videos(violence_test,  "test",  "Violence")

# NonViolence
copy_videos(nonviolence_train, "train", "NonViolence")
copy_videos(nonviolence_val,   "val",   "NonViolence")
copy_videos(nonviolence_test,  "test",  "NonViolence")

print("Dataset split and copied to structured folders.")

# ----------------------------
# 4️ Verify counts
# ----------------------------
for split in ["train", "val", "test"]:
    v_count = len(os.listdir(os.path.join(VIDEOS_DIR, split, "Violence")))
    nv_count = len(os.listdir(os.path.join(VIDEOS_DIR, split, "NonViolence")))
    print(f"{split} - Violence: {v_count}, NonViolence: {nv_count}")

Violence videos: 3500
NonViolence videos: 3500
Dataset split and copied to structured folders.
train - Violence: 2450, NonViolence: 2450
val - Violence: 525, NonViolence: 525
test - Violence: 525, NonViolence: 525


## src/utils/video_utils.py

In [6]:
import os
from configs.config import BASE_DIR 
os.makedirs(BASE_DIR, exist_ok=True)
video_utils_content =''' # video utils for  Violence Detection Project
import cv2
import numpy as np
import torch
from configs.config import NUM_FRAMES, FRAME_SIZE, IMAGENET_MEAN, IMAGENET_STD

def load_video_frames(video_path, num_frames=NUM_FRAMES, resize=FRAME_SIZE):
    import cv2
    import numpy as np
    import torch
    from configs.config import IMAGENET_MEAN, IMAGENET_STD

    frames = []
    last_frame = np.zeros((resize[1], resize[0], 3), dtype=np.uint8)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"⚠️ Cannot open video: {video_path}")
        return torch.zeros((num_frames, 3, resize[1], resize[0]), dtype=torch.float)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        print(f"⚠️ Video has 0 frames: {video_path}")
        return torch.zeros((num_frames, 3, resize[1], resize[0]), dtype=torch.float)

    frame_idxs = np.linspace(0, max(total_frames-1,0), num_frames).astype(int)

    for idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret or frame is None:
            frame = last_frame
        else:
            try:
                # If grayscale, convert to RGB
                if frame.ndim == 2:
                    frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
                # If frame has wrong number of channels
                elif frame.shape[2] != 3:
                    frame = last_frame
                else:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, resize)
            except:
                frame = last_frame

        last_frame = frame
        frame = frame.astype(np.float32)/255.0
        frame = (frame - IMAGENET_MEAN)/IMAGENET_STD
        frame = np.transpose(frame, (2,0,1))
        frames.append(frame)

    cap.release()
    return torch.tensor(np.array(frames), dtype=torch.float)
    
    '''
with open(os.path.join(BASE_DIR , "src/utils/video_utils.py"),"w")as f:
    f.write(video_utils_content )

print("video utils file is ready!")



video utils file is ready!


In [7]:
import torch
import sys
sys.path.append("/kaggle/working/Violence-detection/src/utils")
from video_utils import load_video_frames  # from src/video_dataset.py

In [8]:
import importlib
import video_utils
importlib.reload(video_utils)
from video_utils import load_video_frames

In [9]:
from video_utils import load_video_frames

video_path = "/kaggle/working/Violence-detection/data/videos/train/Violence/V_1.mp4"
frames = load_video_frames(video_path)
print(frames.shape)  # should be [NUM_FRAMES, 3, 224, 224]

torch.Size([16, 3, 224, 224])


In [10]:
import os
import torch
from video_utils import load_video_frames  # your existing function

def filter_corrupted_videos(video_dir, label_name):
    """
    Scan a folder of videos and keep only the ones that OpenCV can read properly.
    Returns: list of (path, label)
    """
    video_list = []
    folder_path = os.path.join(video_dir, label_name)
    if not os.path.exists(folder_path):
        return []

    for f in os.listdir(folder_path):
        if not f.lower().endswith((".mp4", ".avi")):
            continue
        path = os.path.join(folder_path, f)
        try:
            # Try reading frames
            _ = load_video_frames(path, num_frames=1)  # just one frame for fast check
            video_list.append((path, 1 if label_name=="Violence" else 0))
        except Exception as e:
            print(f"⚠️ Skipping corrupted video: {path}, error: {e}")
    return video_list

# ----------------------------
# Run for all splits
# ----------------------------
train_videos = filter_corrupted_videos("/kaggle/working/Violence-detection/data/videos/train", "Violence") + \
               filter_corrupted_videos("/kaggle/working/Violence-detection/data/videos/train", "NonViolence")

val_videos   = filter_corrupted_videos("/kaggle/working/Violence-detection/data/videos/val", "Violence") + \
               filter_corrupted_videos("/kaggle/working/Violence-detection/data/videos/val", "NonViolence")

test_videos  = filter_corrupted_videos("/kaggle/working/Violence-detection/data/videos/test", "Violence") + \
               filter_corrupted_videos("/kaggle/working/Violence-detection/data/videos/test", "NonViolence")

print(f"Train: {len(train_videos)}, Val: {len(val_videos)}, Test: {len(test_videos)}")

Train: 4900, Val: 1050, Test: 1050


In [11]:
print(train_videos[0])
# ('/kaggle/working/Violence-detection/data/videos/train/Violence/V_1.mp4', 1)

('/kaggle/working/Violence-detection/data/videos/train/Violence/file_002231.mp4', 1)


# cached videos 

## src/dataset/video_dataset.py

In [12]:
import os
from configs.config import BASE_DIR 
os.makedirs(BASE_DIR, exist_ok=True)
video_dataset_content =''' # video dataset for  Violence Detection Project

import os,sys
import torch
sys.path.append("/kaggle/working/Violence-detection/src/utils")
from torch.utils.data import Dataset
from video_utils import load_video_frames

class VideoDataset(Dataset):
    def __init__(self, video_list, num_frames=16, transform=None):
        """
        video_list: list of tuples (video_path, label)
        """
        self.video_list = video_list
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.video_list)

    def __getitem__(self, idx):
        path, label = self.video_list[idx]
        video_tensor = load_video_frames(path, num_frames=self.num_frames)
        if self.transform:
            video_tensor = self.transform(video_tensor)
        return video_tensor, torch.tensor(label, dtype=torch.float), path
        '''
        
with open(os.path.join(BASE_DIR, "src/dataset/video_dataset.py"), "w") as f:
    f.write(video_dataset_content )
print("video dataset file is ready!")


video dataset file is ready!


In [13]:
import torch
import sys
sys.path.append("/kaggle/working/Violence-detection/src/dataset")
from video_dataset import VideoDataset  # from src/video_dataset.py

In [14]:
import importlib
import video_dataset
importlib.reload(video_dataset)
from video_dataset import VideoDataset

## DataLoader Example

In [15]:
import sys, os
from configs.config import VIDEOS_DIR, BATCH_SIZE, NUM_WORKERS
from torch.utils.data import DataLoader

train_dataset = VideoDataset(train_videos)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

# Verify labels & shapes
for batch_videos, batch_labels, paths in train_loader:
    print("Batch size:", batch_videos.shape)   # (batch_size, frames, 3, 224, 224)
    print("Labels:", batch_labels)            # 0 or 1
    print("Paths:", paths)
    break

Batch size: torch.Size([4, 16, 3, 224, 224])
Labels: tensor([1., 1., 0., 1.])
Paths: ('/kaggle/working/Violence-detection/data/videos/train/Violence/V_954.mp4', '/kaggle/working/Violence-detection/data/videos/train/Violence/V_982.mp4', '/kaggle/working/Violence-detection/data/videos/train/NonViolence/file_001292.avi', '/kaggle/working/Violence-detection/data/videos/train/Violence/V_148.mp4')


In [40]:
import os

BASE_DIR = "/kaggle/working/Violence-detection/"

total_files = 0
total_size = 0

for root, dirs, files in os.walk(BASE_DIR):
    total_files += len(files)
    total_size += sum(os.path.getsize(os.path.join(root, f)) for f in files)

print(f"Files in working dir: {total_files}")
print(f"Total size: {total_size/1e9:.2f} GB")

Files in working dir: 7007
Total size: 17.40 GB


# 02_vit_model_baseline

In [17]:
import os
import sys
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [18]:
import sys, os

BASE_DIR = "/kaggle/working/Violence-detection"

sys.path.append(BASE_DIR)  # 👈 this is key
sys.path.append(os.path.join(BASE_DIR, "src"))

from configs.config import VIDEOS_DIR, BATCH_SIZE, NUM_WORKERS, NUM_FRAMES

In [19]:
##this use after creating the files of utils and datasets and before iport the functions 
import sys
sys.path.append("/kaggle/working/Violence-detection/src/dataset")
sys.path.append("/kaggle/working/Violence-detection/src/utils")
from video_dataset import VideoDataset  # from src/video_dataset.py
from video_utils import load_video_frames  # from utils/video_utils.py

##  Verify Dataset & DataLoader

In [20]:
# ===============================
# 1️⃣ Verify Dataset & DataLoader
# ===============================
# Train, Val, Test
train_dataset = VideoDataset(train_videos)
val_dataset   = VideoDataset(val_videos )

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
# Check a single batch
for videos, labels, paths in train_loader:
    print("Video batch shape:", videos.shape)   # Expect: [B, NUM_FRAMES, 3, 224, 224]
    print("Labels:", labels)
    print("Paths:", paths[:5])
    break

Train samples: 4900, Val samples: 1050
Video batch shape: torch.Size([4, 16, 3, 224, 224])
Labels: tensor([0., 0., 0., 0.])
Paths: ('/kaggle/working/Violence-detection/data/videos/train/NonViolence/file_001963.avi', '/kaggle/working/Violence-detection/data/videos/train/NonViolence/file_001403.avi', '/kaggle/working/Violence-detection/data/videos/train/NonViolence/file_001171.avi', '/kaggle/working/Violence-detection/data/videos/train/NonViolence/NV_345.mp4')


## Simple ViT Model for Video

In [21]:
# ===============================
# 2️⃣ Define a Simple ViT Model for Video
# ===============================

# For simplicity, we will flatten frames and treat each frame as a patch input to a ViT
# Note: This is a baseline. Later you can use pretrained ViT or video ViT (TimeSformer / Video Swin)

class SimpleViTVideoClassifier(nn.Module):
    def __init__(self, num_frames=NUM_FRAMES, num_classes=1, hidden_dim=256):
        super().__init__()
        self.num_frames = num_frames
        
        # Flatten each frame: [3,224,224] -> linear projection
        self.frame_embed = nn.Linear(3*224*224, hidden_dim)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        # Classification head
        self.classifier = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x):
        # x shape: [B, F, C, H, W]
        B, F, C, H, W = x.shape
        x = x.view(B, F, C*H*W)
        x = self.frame_embed(x)
        x = x.permute(1,0,2)
        x = self.transformer(x)
        x = x.mean(dim=0)
        logits = self.classifier(x).squeeze(1)
        return logits

### Define Loss, Optimizer, Device

In [22]:
# ===============================
# 3️⃣ Define Loss, Optimizer, Device
# ===============================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = SimpleViTVideoClassifier(num_frames=NUM_FRAMES).to(device)
criterion = nn.BCEWithLogitsLoss()   # Binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

Using device: cuda


##  Training & Validation Loop

In [23]:
# ===============================
# 4️⃣ Training & Validation functions
# ===============================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for videos, labels, _ in dataloader:
        videos = videos.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * videos.size(0)
    return total_loss / len(dataloader.dataset)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for videos, labels, _ in dataloader:
            videos = videos.to(device)
            labels = labels.to(device)
            outputs = model(videos)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * videos.size(0)
            preds = torch.sigmoid(outputs) > 0.5
            correct += (preds == labels.byte()).sum().item()
    return total_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

###  Run Baseline Training

In [24]:
# ===============================
# 5️⃣ Run Baseline Training
# ===============================
import matplotlib.pyplot as plt


NUM_EPOCHS = 13  # baseline small run
best_val_acc = 0

# Lists to store metrics
train_losses = []
val_losses = []
val_accuracies = []

# Inside your training loop, after each epoch:
for epoch in range(1, NUM_EPOCHS+1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    # Save metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Save checkpoint if best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(BASE_DIR, "checkpoints", "vit_baselinee.pth"))
        print("✅ Checkpoint saved.")

[h264 @ 0x571381c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x571381c0] error while decoding MB 98 31
[h264 @ 0x571381c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x571381c0] error while decoding MB 98 31
[h264 @ 0x571381c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x571381c0] error while decoding MB 98 31
[h264 @ 0x57746b40] mb_type 104 in P slice too large at 98 31
[h264 @ 0x57746b40] error while decoding MB 98 31
[h264 @ 0x57746b40] mb_type 104 in P slice too large at 98 31
[h264 @ 0x57746b40] error while decoding MB 98 31
[h264 @ 0x57746b40] mb_type 104 in P slice too large at 98 31
[h264 @ 0x57746b40] error while decoding MB 98 31


Epoch 1 | Train Loss: 0.7069 | Val Loss: 0.6910 | Val Acc: 0.5571
✅ Checkpoint saved.


[h264 @ 0x59a513c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a513c0] error while decoding MB 98 31
[h264 @ 0x59a513c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a513c0] error while decoding MB 98 31
[h264 @ 0x59a513c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a513c0] error while decoding MB 98 31
[h264 @ 0x59b72680] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b72680] error while decoding MB 98 31
[h264 @ 0x59b72680] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b72680] error while decoding MB 98 31
[h264 @ 0x59b72680] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b72680] error while decoding MB 98 31


Epoch 2 | Train Loss: 0.6935 | Val Loss: 0.6879 | Val Acc: 0.5000


[h264 @ 0x59a85840] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a85840] error while decoding MB 98 31
[h264 @ 0x59a85840] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a85840] error while decoding MB 98 31
[h264 @ 0x59a85840] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a85840] error while decoding MB 98 31
[h264 @ 0x59a82c40] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a82c40] error while decoding MB 98 31
[h264 @ 0x59a82c40] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a82c40] error while decoding MB 98 31
[h264 @ 0x59a82c40] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a82c40] error while decoding MB 98 31


...

[h264 @ 0x59b3b8c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b3b8c0] error while decoding MB 98 31
[h264 @ 0x59b3b8c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b3b8c0] error while decoding MB 98 31
[h264 @ 0x59b3b8c0] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b3b8c0] error while decoding MB 98 31
[h264 @ 0x59a9a300] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a9a300] error while decoding MB 98 31
[h264 @ 0x59a9a300] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a9a300] error while decoding MB 98 31
[h264 @ 0x59a9a300] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59a9a300] error while decoding MB 98 31


Epoch 8 | Train Loss: 0.6945 | Val Loss: 0.6932 | Val Acc: 0.5000


[h264 @ 0x59ccb800] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59ccb800] error while decoding MB 98 31
[h264 @ 0x59ccb800] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59ccb800] error while decoding MB 98 31
[h264 @ 0x59ccb800] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59ccb800] error while decoding MB 98 31
[h264 @ 0x59b92d00] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b92d00] error while decoding MB 98 31
[h264 @ 0x59b92d00] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b92d00] error while decoding MB 98 31
[h264 @ 0x59b92d00] mb_type 104 in P slice too large at 98 31
[h264 @ 0x59b92d00] error while decoding MB 98 31


Epoch 9 | Train Loss: 0.6942 | Val Loss: 0.6932 | Val Acc: 0.5000


KeyboardInterrupt: 

In [ ]:
# ===============================
# Plot Training & Validation Metrics
# ===============================
import matplotlib.pyplot as plt

# Ensure you have these lists from training
# train_losses, val_losses, val_accuracies

plt.figure(figsize=(12,5))

# Loss plot
plt.subplot(1,2,1)
plt.plot(range(1, len(train_losses)+1), train_losses, label="Train Loss")
plt.plot(range(1, len(val_losses)+1), val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()

# Accuracy plot
plt.subplot(1,2,2)
plt.plot(range(1, len(val_accuracies)+1), val_accuracies, label="Val Accuracy", color='green')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy over Epochs")
plt.legend()

plt.show()

## Load Checkpoint & Test 

In [25]:
# ===============================
# 6️⃣ Test Skeleton
# ===============================

test_dataset  = VideoDataset(test_videos)

test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

model.load_state_dict(torch.load(os.path.join(BASE_DIR, "checkpoints", "vit_baselinee.pth")))
model.to(device)
model.eval()

test_correct = 0
with torch.no_grad():
    for videos, labels, _ in test_loader:
        videos = videos.to(device)
        labels = labels.to(device)
        outputs = model(videos)
        preds = torch.sigmoid(outputs) > 0.5
        test_correct += (preds == labels.byte()).sum().item()

test_acc = test_correct / len(test_dataset)
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.5733


In [34]:
import os
import torch

# Save model in a folder
checkpoint_dir = "/kaggle/working/vit_model"
os.makedirs(checkpoint_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(checkpoint_dir, "vit_baseline.pth"))

In [39]:
import torch

file1 = "/kaggle/working/Violence-detection/checkpoints/vit_baselinee.pth"
file2 = "/kaggle/working/vit_model/vit_baselinee.pth"

state1 = torch.load(file1)
state2 = torch.load(file2)

# Compare all keys
all_same = all(torch.equal(state1[k], state2[k]) for k in state1)
print("Are the weights identical?", all_same)

Are the weights identical? True
